### Processing Incremental Updates with Structured Streaming and Delta Lake
In this lab you'll apply your knowledge of structured streaming and Auto Loader to implement a simple multi-hop architecture.

#### 1.0. Import Shared Utilities and Data Files
Run the following cell to setup necessary variables and clear out past runs of this notebook. Note that re-executing this cell will allow you to start the lab over.

In [0]:
%run ./Includes/5.1-Lab-setup


Creating the database "dbacademy_briceyhowe_gmail_com_dewd_5_1"

Predefined Paths:
  DA.paths.working_dir: dbfs:/user/briceyhowe@gmail.com/dbacademy/dewd/5.1
  DA.paths.user_db:     dbfs:/user/briceyhowe@gmail.com/dbacademy/dewd/5.1/5_1.db
  DA.paths.checkpoints: dbfs:/user/briceyhowe@gmail.com/dbacademy/dewd/5.1/_checkpoints

Predefined tables in dbacademy_briceyhowe_gmail_com_dewd_5_1:
  -none-

Setup completed in 2 seconds



#### 2.0. Bronze Table: Ingest data
This lab uses a collection of customer-related CSV data from DBFS found in *`dbfs:/FileStore/lab_data/retail-org/customers/`*.
- Read this data using Auto Loader using its schema inference (use **`DA.paths.checkpoints`** to store the schema info in a dedicated folder for **`customers`**).
- Stream the raw data to a Delta table called **`bronze`** using the **`append`** output mode.

In [0]:
dbutils.fs.cp(
  "dbfs:/databricks-datasets/retail-org/customers/",
  "dbfs:/FileStore/lab_data/retail-org/customers/",
  recurse=True
)


True

In [0]:
customerDF = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", f"{DA.paths.checkpoints}/customers_schema")
    .load("dbfs:/FileStore/lab_data/retail-org/customers/")
)

bronzeQuery = (customerDF.writeStream
    .format("delta")
    .option("checkpointLocation", f"{DA.paths.checkpoints}/bronze_customers")
    .outputMode("append")
    .start("dbfs:/FileStore/lab_data/retail-org/bronze_customers/")
)


In [0]:
DA.block_until_stream_is_ready(bronzeQuery)

The stream has processed 2 batchs


##### 2.1. Create a Streaming Temporary View
Create a streaming temporary view named **`bronze_temp`** into the **`bronze`** table so we can perform transformations using SQL.

In [0]:
(spark
  .readStream
  .format("delta")
  .load("dbfs:/FileStore/lab_data/retail-org/bronze_customers/")
  .createOrReplaceTempView("bronze_temp"))


In [0]:
%sql
SELECT * FROM bronze_temp
LIMIT 10


customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment,_rescued_data
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3,null
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3,null
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,null,5.0,0,null
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195.0,7.0,1,null
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,null,0.0,0,null
20441596,null,null,"TIRADO, MARCO A",NY,Otselic,13072,County Road 16,2792,null,NY,Chenango,-75.7505808,42.7172722,"NY, 13072, County Road 16, 2792",1519335250,null,24.0,3,null
5945686,null,null,"SKORA, BRIAN S",MI,null,48205.0,E 8 MILE RD,16414.0,null,null,null,-82.950874,42.4499233,"MI, 48205.0, E 8 MILE RD, 16414.0",1518988242,null,7.0,1,null
5385771,null,null,"SLAWEK, DEAN J",PA,null,19147-3204,FITZWATER ST,328,null,null,null,-75.14920550000002,39.9389473,"PA, 19147-3204, FITZWATER ST, 328",1518239268,null,18.0,3,null
1427940,null,null,"REAVES, LIONEL C",VA,HOT SPRINGS,24445.0,HOT SPRINGS RD,6419.0,null,null,null,-79.90497859999998,37.8949737,"VA, 24445.0, HOT SPRINGS RD, 6419.0",1529087690,null,10.0,2,null
10457387,null,null,"BONGIOVANNI, KELLY M",IN,VINCENNES,47591,JERRY ST,2006.0,null,Indiana,42.0,-87.519002,38.662178,"IN, 47591, JERRY ST, 2006.0",1535887733,null,9.0,2,null


##### 2.2. Clean and Enhance the Data
Use the CTAS syntax to define a new streaming view called **`bronze_enhanced_temp`** that does the following:
* Skips records with a null **`postcode`** (set to zero)
* Inserts a column called **`receipt_time`** containing a current timestamp
* Inserts a column called **`source_file`** containing the input filename

In [0]:
%sql
-- TODO:
CREATE OR REPLACE TEMPORARY VIEW bronze_enhanced_temp AS
SELECT 
  *,
  current_timestamp() AS receipt_time,
  input_file_name() AS source_file
FROM bronze_temp
WHERE postcode IS NOT NULL


In [0]:
%sql
SELECT * FROM bronze_enhanced_temp
LIMIT 10

customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment,_rescued_data,receipt_time,source_file
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,null,5.0,0,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195.0,7.0,1,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,null,0.0,0,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
20441596,null,null,"TIRADO, MARCO A",NY,Otselic,13072,County Road 16,2792,null,NY,Chenango,-75.7505808,42.7172722,"NY, 13072, County Road 16, 2792",1519335250,null,24.0,3,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
5945686,null,null,"SKORA, BRIAN S",MI,null,48205.0,E 8 MILE RD,16414.0,null,null,null,-82.950874,42.4499233,"MI, 48205.0, E 8 MILE RD, 16414.0",1518988242,null,7.0,1,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
5385771,null,null,"SLAWEK, DEAN J",PA,null,19147-3204,FITZWATER ST,328,null,null,null,-75.14920550000002,39.9389473,"PA, 19147-3204, FITZWATER ST, 328",1518239268,null,18.0,3,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
1427940,null,null,"REAVES, LIONEL C",VA,HOT SPRINGS,24445.0,HOT SPRINGS RD,6419.0,null,null,null,-79.90497859999998,37.8949737,"VA, 24445.0, HOT SPRINGS RD, 6419.0",1529087690,null,10.0,2,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
10457387,null,null,"BONGIOVANNI, KELLY M",IN,VINCENNES,47591,JERRY ST,2006.0,null,Indiana,42.0,-87.519002,38.662178,"IN, 47591, JERRY ST, 2006.0",1535887733,null,9.0,2,null,2025-04-07T19:12:38.079Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet


#### 3.0. Silver Table
Stream the data from **`bronze_enhanced_temp`** to a Delta table named **`silver`** using the **`append`** output mode. Use **`DA.paths.checkpoints`** and a dedicated folder for **`silver`** as the checkpoint path).

In [0]:
# TODO:
silverQuery = (spark.table("bronze_enhanced_temp")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{DA.paths.checkpoints}/silver_customers")
    .start("dbfs:/FileStore/lab_data/retail-org/silver_customers/")
)

In [0]:
DA.block_until_stream_is_ready(silverQuery)

The stream has processed 2 batchs


##### 3.1. Create a Streaming Temporary View
Create another streaming temporary view named **`silver_temp`** for the **`silver`** table so we can perform business-level queries using SQL.

In [0]:
(spark.readStream
  .format("delta")
  .load("dbfs:/FileStore/lab_data/retail-org/silver_customers/")
  .createOrReplaceTempView("silver_temp"))


In [0]:
%sql
SELECT * FROM silver_temp
LIMIT 10


customer_id,tax_id,tax_code,customer_name,state,city,postcode,street,number,unit,region,district,lon,lat,ship_to_address,valid_from,valid_to,units_purchased,loyalty_segment,_rescued_data,receipt_time,source_file
11123757,null,null,"SMITH, SHIRLEY",IN,BREMEN,46506.0,N CENTER ST,521.0,null,Indiana,50.0,-86.1465825,41.4507625,"IN, 46506.0, N CENTER ST, 521.0",1532824233,1548137353.0,34.0,3,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
30585978,null,null,"STEPHENS, GERALDINE M",OR,ADDRESS,0,NO SITUS,null,null,null,null,-122.1055158,45.374317,"OR, 0, NO SITUS, nan",1523100473,null,18.0,3,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
349822,null,null,"GUZMAN, CARMEN",VA,VIENNA,22181,HILL RD,2860,null,VA,null,-77.2941261,38.88303270000001,"VA, 22181, HILL RD, 2860",1522922493,null,5.0,0,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
27652636,null,null,"HASSETT, PATRICK J",WI,VILLAGE OF NASHOTAH,53058.0,IVY LANE,W333N 5591,null,null,null,-88.40951700000002,43.1213789,"WI, 53058.0, IVY LANE, W333N 5591",1531834357,1558052195.0,7.0,1,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
14437343,null,null,"HENTZ, DIANA L",OH,COLUMBUS,43228.0,ALLIANCE WAY,5706,null,OH,FRA,-83.158438,39.97821810000001,"OH, 43228.0, ALLIANCE WAY, 5706",1517227530,null,0.0,0,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
20441596,null,null,"TIRADO, MARCO A",NY,Otselic,13072,County Road 16,2792,null,NY,Chenango,-75.7505808,42.7172722,"NY, 13072, County Road 16, 2792",1519335250,null,24.0,3,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
5945686,null,null,"SKORA, BRIAN S",MI,null,48205.0,E 8 MILE RD,16414.0,null,null,null,-82.950874,42.4499233,"MI, 48205.0, E 8 MILE RD, 16414.0",1518988242,null,7.0,1,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
5385771,null,null,"SLAWEK, DEAN J",PA,null,19147-3204,FITZWATER ST,328,null,null,null,-75.14920550000002,39.9389473,"PA, 19147-3204, FITZWATER ST, 328",1518239268,null,18.0,3,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
1427940,null,null,"REAVES, LIONEL C",VA,HOT SPRINGS,24445.0,HOT SPRINGS RD,6419.0,null,null,null,-79.90497859999998,37.8949737,"VA, 24445.0, HOT SPRINGS RD, 6419.0",1529087690,null,10.0,2,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet
10457387,null,null,"BONGIOVANNI, KELLY M",IN,VINCENNES,47591,JERRY ST,2006.0,null,Indiana,42.0,-87.519002,38.662178,"IN, 47591, JERRY ST, 2006.0",1535887733,null,9.0,2,null,2025-04-07T18:53:25.67Z,dbfs:/FileStore/lab_data/retail-org/bronze_customers/part-00000-00521cd5-075e-45a2-88d9-e7b491fd2dfb.c000.snappy.parquet



#### 4.0. Gold Table
Use the CTAS syntax to define a new streaming view called **`customer_count_by_state_temp`** that counts customers per state.

In [0]:
%sql
-- TODO:
CREATE OR REPLACE TEMP VIEW customer_count_by_state_temp AS
SELECT state, COUNT(*) AS customer_count
FROM silver_temp
GROUP BY state


In [0]:
%sql
SELECT * FROM customer_count_by_state_temp
LIMIT 10

state,customer_count
IL,8760
SD,260
TN,1400
NJ,15030
SC,1730
ID,320
AZ,6000
LA,1110
WY,260
OK,300


Finally, stream the data from the **`customer_count_by_state_temp`** view to a Delta table called **`gold_customer_count_by_state`**. Remember to use the **`complete`** output mode because aggregations like **`count()`** and sorting cannot operate on *unbounded* datasets.  Also, use **`DA.paths.checkpoints`** and a dedicated folder for **`customer_counts`** as the checkpoint path).

In [0]:
# TODO:

goldQuery = (spark.table("customer_count_by_state_temp")
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("mergeSchema", "true")
    .option("checkpointLocation", f"{DA.paths.checkpoints}/customer_counts")
    .start("dbfs:/FileStore/lab_data/retail-org/gold_customer_count_by_state/")
)


In [0]:
DA.block_until_stream_is_ready(goldQuery)

The stream has processed 2 batchs


#### 5.0. Query the Results
Query the **`gold_customer_count_by_state`** table (this will not be a streaming query).

In [0]:
%sql
CREATE TABLE gold_customer_count_by_state
USING DELTA
LOCATION 'dbfs:/FileStore/lab_data/retail-org/gold_customer_count_by_state/'


In [0]:
%sql
SELECT * FROM gold_customer_count_by_state
LIMIT 10

state,customer_count
MT,2030
TX,5670
NV,400
AR,110
NH,20
WI,9920
VT,1570
HI,650
UT,4160
AL,650


#### 6.0. Clean Up
Run the following cell to remove the database and all data associated with this lab.

In [0]:
DA.cleanup()

Stopping the stream "None"
Stopping the stream "display_query_15"
Stopping the stream "display_query_12"
Stopping the stream "None"
Stopping the stream "None"
Stopping the stream "display_query_14"
Stopping the stream "display_query_13"
Dropping the database "dbacademy_briceyhowe_gmail_com_dewd_5_1"
Removing the working directory "dbfs:/user/briceyhowe@gmail.com/dbacademy/dewd/5.1"


By completing this lab, you should now feel comfortable:
* Using PySpark to configure Auto Loader for incremental data ingestion
* Using Spark SQL to aggregate streaming data
* Streaming data to a Delta table